# 02 · Train a quadratic curve explicitly
We fit `lactate = c + a·power + b·power²` using `PolynomialFeatures(degree=2)` plus ordinary least squares. A degree-1 straight line is the baseline. Entire test files are reserved for the final test set.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "data" / "processed" / "cycling_lactate.csv"
assert DATA.exists(), f"Dataset missing: {DATA}"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 20)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
raw=pd.read_csv(DATA)
import sys
sys.path.insert(0,str(ROOT / "backend"))
from app.model import validate
df=validate(raw)
train_i,test_i=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=42).split(df,groups=df.athlete_id))
train_df,test_df=df.iloc[train_i],df.iloc[test_i]
assert set(train_df.athlete_id).isdisjoint(test_df.athlete_id)
print(f"Train: {len(train_df)} rows / {train_df.athlete_id.nunique()} files")
print(f"Test: {len(test_df)} rows / {test_df.athlete_id.nunique()} files")

## Fit both models on training files only
Polynomial feature generation transforms `power` into two features: `power` and `power²`. The linear model sees only `power`.

In [ ]:
X_train=train_df[["training_load"]]
y_train=train_df.performance_score
X_test=test_df[["training_load"]]
y_test=test_df.performance_score
poly=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),LinearRegression()).fit(X_train,y_train)
line=make_pipeline(PolynomialFeatures(degree=1,include_bias=False),LinearRegression()).fit(X_train,y_train)
c=poly.named_steps["linearregression"]
print(f"Fitted equation: lactate = {c.intercept_:.4f} + ({c.coef_[0]:.6f} × watts) + ({c.coef_[1]:.8f} × watts²)")

## Evaluate on unseen test files
MAE is the average absolute error in **mmol/L**; R² compares squared error to a constant test-mean predictor. The test files are not used for model choice.

In [ ]:
pred_poly=poly.predict(X_test)
pred_line=line.predict(X_test)
results=pd.DataFrame({"model":["Linear degree 1","Polynomial degree 2"],
"MAE mmol/L":[mean_absolute_error(y_test,pred_line),mean_absolute_error(y_test,pred_poly)],
"RMSE mmol/L":[np.sqrt(mean_squared_error(y_test,pred_line)),np.sqrt(mean_squared_error(y_test,pred_poly))],
"R²":[r2_score(y_test,pred_line),r2_score(y_test,pred_poly)]})
print(results.round(3))

## Inspect the fit and residuals
Large spread around the pooled curve can reflect individual physiology, sport level and noisy measurements. Residuals are `measured − predicted`.

In [ ]:
grid=np.linspace(X_train.training_load.min(),X_train.training_load.max(),150)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(X_train.training_load,y_train,s=8,alpha=.12,label="Training stages")
axes[0].scatter(X_test.training_load,y_test,s=12,alpha=.3,label="Held-out stages")
axes[0].plot(grid,poly.predict(pd.DataFrame({"training_load":grid})),color="#34a853",lw=3,label="Quadratic")
axes[0].plot(grid,line.predict(pd.DataFrame({"training_load":grid})),color="#f2994a",ls="--",lw=2,label="Linear")
axes[0].set(xlabel="Cycling power (W)",ylabel="Blood lactate (mmol/L)",title="Training fit and held-out observations")
axes[0].legend()
axes[1].scatter(pred_poly,y_test-pred_poly,s=14,alpha=.4)
axes[1].axhline(0,color="black",ls="--")
axes[1].set(xlabel="Quadratic prediction (mmol/L)",ylabel="Residual (measured − predicted)",title="Held-out residuals")
plt.tight_layout();plt.show()

## Interpretation
A better R² than the line supports a curved *pooled association*. It does not establish a safe workload, a clinical lactate threshold, or a forecast of an athlete’s future performance. Predict only within the training workload range.